# Treino incremental de ML com **Spark Streaming** + **MinIO (S3 open source)** — Base de Diabetes

**Objetivo:** treinar um classificador de **diabetes** de forma **incremental (online learning)** conforme os dados chegam por um fluxo (**Spark Structured Streaming**), salvando cada checkpoint do modelo no **MinIO** — a implementação **open source compatível com o Amazon S3** — e mostrando, a cada micro-batch, a **acurácia** e a **superfície de separação** (fronteira de decisão) evoluindo ao longo do treino.

| Etapa | Tecnologia |
|-------|------------|
| Dataset | Pima Indians Diabetes (768 pacientes) |
| Fonte do stream | pasta monitorada (chegada incremental de pacientes) |
| Processamento | **Apache Spark Structured Streaming** |
| Modelo | `SGDClassifier` (scikit-learn) com `partial_fit` |
| Armazenamento | **MinIO** (S3 open source) — modelos, métricas e gráficos |
| Visualização | acurácia + superfície de separação por batch |

> ⚠️ **Runtime → Factory reset runtime** antes de executar, para garantir uma VM limpa.

### Por que a fronteira "se move"?
O `SGDClassifier` faz **aprendizado incremental**: a cada micro-batch chamamos `partial_fit`, que dá alguns passos de gradiente com os novos pacientes. Assim a **superfície de separação** vai se ajustando e a **acurácia** tende a subir conforme mais dados chegam — e cada estado do modelo é persistido no MinIO/S3.

## 1. Instalar e iniciar o MinIO (S3 open source) local no Colab

Baixamos o servidor `minio` e o cliente `mc`, e subimos o serviço em `127.0.0.1:9000`.

In [ ]:
%%bash
set -e

echo "=== Baixando MinIO Server + Client ==="
cd /content
wget -q https://dl.min.io/server/minio/release/linux-amd64/minio -O /usr/local/bin/minio
wget -q https://dl.min.io/client/mc/release/linux-amd64/mc -O /usr/local/bin/mc
chmod +x /usr/local/bin/minio /usr/local/bin/mc

mkdir -p /content/minio_data
export MINIO_ROOT_USER=minioadmin
export MINIO_ROOT_PASSWORD=minioadmin

echo "=== Iniciando MinIO em 127.0.0.1:9000 (console 9001) ==="
nohup minio server /content/minio_data \
  --address 127.0.0.1:9000 \
  --console-address 127.0.0.1:9001 \
  > /content/minio.log 2>&1 &
sleep 8
tail -n 3 /content/minio.log
echo "✅ MinIO (S3) rodando em http://127.0.0.1:9000"


## 2. Dependências Python + configuração do S3 (MinIO)

In [ ]:
%pip -q install pyspark==3.5.1 scikit-learn boto3 pandas matplotlib

import io
import json
import time
import pickle
import threading
from pathlib import Path
from datetime import datetime

import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# S3 / MinIO — TUDO LOCAL NA VM DO COLAB
# ============================================================
S3_ENDPOINT = "http://127.0.0.1:9000"
S3_KEY = "minioadmin"
S3_SECRET = "minioadmin"
S3_BUCKET = "diabetes-training"

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=S3_KEY,
    aws_secret_access_key=S3_SECRET,
    region_name="us-east-1",
)

# Cria o bucket (idempotente)
existentes = [b["Name"] for b in s3.list_buckets().get("Buckets", [])]
if S3_BUCKET not in existentes:
    s3.create_bucket(Bucket=S3_BUCKET)

STREAM_DIR = Path("/content/diabetes_stream")
STREAM_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 55)
print("  S3 (MinIO):", S3_ENDPOINT)
print("  Bucket:    ", S3_BUCKET)
print("  Buckets:   ", [b["Name"] for b in s3.list_buckets()["Buckets"]])
print("=" * 55)


## 3. Carregar a base de Diabetes e preparar treino/teste

Usamos a base **Pima Indians Diabetes**. Para poder desenhar a **superfície de separação** em 2D, treinamos com duas variáveis muito informativas: **Glicose** e **IMC (BMI)**. Um conjunto de **teste fixo** é separado para medir a acurácia de forma justa a cada batch.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
COLS = ["preg", "glucose", "bp", "skin", "insulin", "bmi", "pedigree", "age", "outcome"]
dados = pd.read_csv(URL, header=None, names=COLS)

# Glicose e BMI = 0 são valores ausentes; removemos para o exemplo
dados = dados[(dados["glucose"] > 0) & (dados["bmi"] > 0)].reset_index(drop=True)

X = dados[["glucose", "bmi"]].values.astype(float)
y = dados["outcome"].values.astype(int)

# Conjunto de teste FIXO para medir a acurácia ao longo do treino
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Scaler ajustado no treino (padroniza glicose/BMI para o SGD convergir)
scaler = StandardScaler().fit(X_train)
X_test_s = scaler.transform(X_test)

print(f"Total: {len(dados)} pacientes | treino: {len(X_train)} | teste: {len(X_test)}")
print(f"Proporção diabéticos (treino): {y_train.mean():.2%}")
dados.head()


## 4. Produtor — chegada incremental de pacientes (stream)

Embaralhamos o conjunto de treino e gravamos os pacientes em **lotes** (arquivos JSON Lines) numa pasta monitorada pelo Spark, simulando a chegada contínua de novos exames.

In [ ]:
CHUNK = 30           # pacientes por lote
INTERVALO_SEG = 3    # intervalo entre lotes

# Embaralha a ordem de chegada
rng = np.random.default_rng(7)
ordem = rng.permutation(len(X_train))


def produzir_stream(stop_event: threading.Event):
    lote = 0
    for ini in range(0, len(ordem), CHUNK):
        if stop_event.is_set():
            break
        lote += 1
        idxs = ordem[ini:ini + CHUNK]
        ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
        linhas = []
        for i in idxs:
            linhas.append(json.dumps({
                "glucose": float(X_train[i, 0]),
                "bmi": float(X_train[i, 1]),
                "outcome": int(y_train[i]),
            }))
        arq = STREAM_DIR / f"lote{lote:03d}_{ts}.json"
        arq.write_text("\n".join(linhas), encoding="utf-8")
        print(f"🩸 {arq.name} → {len(idxs)} pacientes")
        time.sleep(INTERVALO_SEG)
    print("🏁 Produtor encerrado.")


print(f"Serão gerados ~{int(np.ceil(len(ordem) / CHUNK))} lotes de até {CHUNK} pacientes.")


## 5. Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType

spark = (
    SparkSession.builder
    .appName("DiabetesStreamingTrainMinIO")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}")


## 6. Streaming + treino incremental + gravação no S3 (MinIO)

A cada micro-batch o Spark entrega um lote de pacientes. No `foreachBatch`:

1. Convertemos para pandas e padronizamos (glicose, BMI).
2. Chamamos **`model.partial_fit`** (aprendizado incremental).
3. Medimos a **acurácia** no conjunto de teste fixo.
4. Salvamos no **MinIO/S3**: o **modelo** (`.pkl`), as **métricas** (`.json`) e o **gráfico da superfície de separação** (`.png`).
5. Exibimos a fronteira de decisão atual.

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score

# Estado global do treino incremental
model = SGDClassifier(loss="log_loss", alpha=1e-3, random_state=42)
_estado = {"init": False, "batches": 0, "vistos": 0}
historico = []   # (batch, pacientes_vistos, acuracia)


def salvar_s3(key: str, data: bytes, content_type: str):
    s3.put_object(Bucket=S3_BUCKET, Key=key, Body=data, ContentType=content_type)


def plot_superficie(batch_id: int, acc: float) -> bytes:
    # Malha no espaço padronizado (glicose, BMI)
    x_min, x_max = X_test_s[:, 0].min() - 1, X_test_s[:, 0].max() + 1
    y_min, y_max = X_test_s[:, 1].min() - 1, X_test_s[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(6.5, 5))
    ax.contourf(xx, yy, Z, alpha=0.25, cmap="coolwarm", levels=1)
    ax.contour(xx, yy, Z, colors="k", linewidths=1, levels=[0.5])
    ax.scatter(X_test_s[y_test == 0, 0], X_test_s[y_test == 0, 1],
               c="#2b6cb0", s=18, edgecolor="w", label="Não diabético")
    ax.scatter(X_test_s[y_test == 1, 0], X_test_s[y_test == 1, 1],
               c="#c53030", s=18, edgecolor="w", label="Diabético")
    ax.set_title(f"Batch {batch_id} | vistos={_estado['vistos']} | acurácia={acc:.1%}")
    ax.set_xlabel("Glicose (padronizada)")
    ax.set_ylabel("IMC / BMI (padronizado)")
    ax.legend(loc="upper left", fontsize=8)
    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=90)
    plt.show()
    plt.close(fig)
    return buf.getvalue()


def treinar_batch(batch_df, batch_id: int):
    pdf = batch_df.toPandas()
    if pdf.empty:
        return

    Xb = scaler.transform(pdf[["glucose", "bmi"]].values)
    yb = pdf["outcome"].values.astype(int)

    if not _estado["init"]:
        model.partial_fit(Xb, yb, classes=np.array([0, 1]))
        _estado["init"] = True
    else:
        model.partial_fit(Xb, yb)

    _estado["batches"] += 1
    _estado["vistos"] += len(yb)
    acc = accuracy_score(y_test, model.predict(X_test_s))
    historico.append((batch_id, _estado["vistos"], acc))

    # 1) Modelo (pickle) -> S3
    salvar_s3(f"models/model_batch{batch_id:03d}.pkl",
              pickle.dumps({"model": model, "scaler": scaler}),
              "application/octet-stream")
    # 2) Métricas -> S3
    metric = {"batch": batch_id, "pacientes_vistos": _estado["vistos"],
              "acuracia": acc, "ts": datetime.utcnow().isoformat()}
    salvar_s3(f"metrics/metrics_batch{batch_id:03d}.json",
              json.dumps(metric).encode(), "application/json")
    # 3) Superfície de separação -> S3 (e exibe)
    png = plot_superficie(batch_id, acc)
    salvar_s3(f"plots/boundary_batch{batch_id:03d}.png", png, "image/png")

    print(f"Batch {batch_id}: acurácia={acc:.1%} | +{len(yb)} pacientes | salvo no S3/MinIO")


schema = StructType([
    StructField("glucose", DoubleType(), False),
    StructField("bmi", DoubleType(), False),
    StructField("outcome", IntegerType(), False),
])

pacientes_df = (
    spark.readStream
    .schema(schema)
    .option("maxFilesPerTrigger", 1)   # 1 arquivo = 1 batch = 1 passo de treino
    .json(str(STREAM_DIR))
)

query = (
    pacientes_df.writeStream
    .outputMode("append")
    .foreachBatch(treinar_batch)
    .option("checkpointLocation", "/content/checkpoint_diabetes")
    .trigger(processingTime="4 seconds")
    .start()
)

print(f"Streaming query id: {query.id}")


## 7. Executar o produtor e treinar ao vivo

O produtor injeta lotes de pacientes; o Spark treina o modelo incrementalmente e, a cada batch, você vê a **superfície de separação** se ajustando e a **acurácia** subindo — com tudo salvo no MinIO/S3.

In [ ]:
stop_event = threading.Event()
produtor = threading.Thread(target=produzir_stream, args=(stop_event,), daemon=True)
produtor.start()

try:
    n_lotes = int(np.ceil(len(ordem) / CHUNK))
    tempo_total = n_lotes * INTERVALO_SEG + 30
    print(f"⏳ Treinando por ~{tempo_total}s...")
    query.awaitTermination(timeout=tempo_total)
finally:
    stop_event.set()
    produtor.join(timeout=10)
    time.sleep(6)
    query.stop()
    print("✅ Treino em streaming finalizado.")


## 8. Evolução da acurácia ao longo do treino

In [ ]:
if not historico:
    print("Nenhum batch treinado. Rode as células anteriores novamente.")
else:
    hist = pd.DataFrame(historico, columns=["batch", "pacientes_vistos", "acuracia"])
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(hist["pacientes_vistos"], hist["acuracia"], marker="o", color="#2f855a")
    ax.set_title("Acurácia no conjunto de teste ao longo do treino incremental")
    ax.set_xlabel("Pacientes vistos (acumulado)")
    ax.set_ylabel("Acurácia")
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()
    display(hist.tail())


## 9. Conferir os artefatos salvos no S3 (MinIO)

Listamos os objetos gravados no bucket e recarregamos o **modelo final** direto do S3.

In [ ]:
objs = s3.list_objects_v2(Bucket=S3_BUCKET).get("Contents", [])
print(f"Total de objetos no bucket '{S3_BUCKET}': {len(objs)}")
for prefixo in ["models/", "metrics/", "plots/"]:
    qtd = sum(1 for o in objs if o["Key"].startswith(prefixo))
    print(f"  {prefixo:9s} → {qtd} arquivo(s)")

# Recarrega o último modelo salvo diretamente do S3
modelos = sorted(o["Key"] for o in objs if o["Key"].startswith("models/"))
if modelos:
    ultimo = modelos[-1]
    body = s3.get_object(Bucket=S3_BUCKET, Key=ultimo)["Body"].read()
    artefato = pickle.loads(body)
    modelo_s3 = artefato["model"]
    scaler_s3 = artefato["scaler"]
    acc_final = accuracy_score(y_test, modelo_s3.predict(scaler_s3.transform(X_test)))
    print(f"\n📦 Modelo recarregado do S3: {ultimo}")
    print(f"   Acurácia final: {acc_final:.1%}")


## 10. Prever um novo paciente com o modelo carregado do S3

In [ ]:
def prever(glicose: float, imc: float):
    Xn = scaler_s3.transform([[glicose, imc]])
    pred = int(modelo_s3.predict(Xn)[0])
    prob = float(modelo_s3.predict_proba(Xn)[0, 1])
    rotulo = "DIABÉTICO" if pred == 1 else "NÃO diabético"
    print(f"Glicose={glicose} | IMC={imc} → {rotulo} (prob. diabetes: {prob:.1%})")


prever(85, 22)     # perfil saudável
prever(180, 38)    # perfil de risco


## 11. (Opcional) Encerrar Spark

In [ ]:
try:
    query.stop()
except Exception:
    pass
spark.stop()
print("🛑 Spark encerrado.")


## 12. Deploy (fora do Colab)

Para rodar em servidor use o **Docker Compose** com **MinIO** e os scripts em [`deploy_diabetes/`](https://github.com/naubergois/SparkSteammingAulas/tree/main/deploy_diabetes):

```bash
cd deploy_diabetes
docker compose up -d                 # sobe o MinIO (S3) em localhost:9000

pip install -r requirements.txt
python producer.py                   # gera o stream de pacientes em ./diabetes_stream

spark-submit stream_train_job.py     # treina incremental e salva no MinIO/S3
```

- **MinIO (S3):** `http://localhost:9000` (console `http://localhost:9001`, `minioadmin`/`minioadmin`)
- **Bucket:** `diabetes-training` (models/ + metrics/ + plots/)
